# Flux — scFEA metabolni flux (TRIM-Flux Var 2)

Izracuna metabolni flux za vsako celico prek **scFEA** (single-cell Flux Estimation Analysis).
Flux postane 3. modaliteta v TRIM (poleg RNA + TCR).

**Vhod:** `data_rna_counts.pkl` (SUROVI counti — scFEA jih sam normalizira; NE normalizirani `data_rna.pkl`)
**Izhod:** `data_flux.pkl` — matrika (celice x ~168 metabolnih modulov), poravnana z `data_labels`

Orodje scFEA (izbrano po raziskavi izvedljivosti): GNN, GPU, per-cell, dropout-robusten (korelacija >0.85),
~168 cloveskih metabolnih modulov. Nevzdrzevan od 2021 -> potrebni patchi za moderni Colab (spodaj).

> scFEA fluksi so RELATIVNI/model-odvisni (ne absolutne hitrosti); benchmark scFEA/Compass/METAFlux ne obstaja.


## 0. Mount + namestitev scFEA (+ patchi za moderni Colab)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone -q https://github.com/changwn/scFEA.git
%cd /content/scFEA

# scFEA importa 'magic' na vrhu skripte -> nujno nalozen (tudi pri sc_imputation=False).
# --no-deps: sicer vlece star pandas iz vira -> build pade.
!pip install -q --no-deps magic-impute graphtools scprep s_gd2 pygsp Deprecated tasklogger wrapt

# PATCHI (scFEA pisan za pandas<2 / stari torch):
# pandas 3: .append IN ._append odstranjena -> pd.concat (dela na pandas 2 in 3)
# Ciljaj TOCNO 'geneExprDf.append(temp,' (DataFrame), NE list.append.
!sed -i 's/geneExprDf = geneExprDf\.append(temp,/geneExprDf = pd.concat([geneExprDf, temp],/' src/scFEA.py
!sed -i 's/\.detach()\.numpy()/.detach().cpu().numpy()/g' src/scFEA.py  # torch: GPU tensor -> cpu

import magic
print('magic:', getattr(magic, '__version__', 'OK'))
print('cmMat na voljo:')
!ls data/ | grep -iE 'module_gene|cmMat'

## 1. Nalozi surove counte + gene imena

scFEA hoce **surove counte** (sam logira ce max>50) in **gene simbole** (vrstice=geni, stolpci=celice).
`data_rna_counts.pkl` je shranjen v notebooku 01 (loceno od normaliziranega `data_rna.pkl`).

In [ ]:
import pickle, numpy as np, pandas as pd, os, time
from scipy.sparse import issparse

DATA = '/content/drive/MyDrive/Diploma/data/processed'
with open(os.path.join(DATA, 'data_rna_counts.pkl'), 'rb') as f:
    counts = pickle.load(f)                 # SUROVI counti (celice x geni), sparse
with open(os.path.join(DATA, 'gene_names.pkl'), 'rb') as f:
    gene_names = [str(g) for g in pickle.load(f)]

print('counts:', counts.shape, type(counts).__name__)
print('gene_names:', len(gene_names), '| primer:', gene_names[:4])
assert len(gene_names) == counts.shape[1], 'gene_names != stolpci counts!'

# scFEA rabi gene SIMBOLE (CD8A), ne Ensembl ID (ENSG...).
is_ensembl = all(g.upper().startswith('ENSG') for g in gene_names[:50])
assert not is_ensembl, 'Geni so Ensembl ID -> scFEA rabi simbole (pretvori z mygene)!'
print('Geni so simboli:', not is_ensembl)

## 2. Batch scFEA -> flux (cel dataset)

70k celic naenkrat crasha RAM (scFEA 'process data' zanka = 168 kopij matrike).
Resitev: kosi po 5000 celic, vsak scFEA locen (zaporedno, GPU je deljiv le tako),
zdruzimo flux. Filtrirano na ~658 modulnih genov (identicen rezultat, hitro).
tqdm pokaze napredek. Izhod: `data_flux.pkl` (celice x ~168 modulov).

In [ ]:
# ============================================================================
# BATCH scFEA: 70k celic naenkrat = "process data" zanka (168 kopij) crasha RAM.
# Resitev: kosi po CHUNK celic, vsak scFEA locen (hiter), zdruzimo flux.
# Batchi ZAPOREDNO (ne vzporedno): scFEA je samostojna skripta ki zasede cel GPU;
# vzporedni klici bi se borili za GPU (OOM). Ozko grlo je CPU/RAM zanka, ne GPU ->
# majhni kosi jo naredijo hitro. tqdm pokaze napredek.
# ============================================================================
from tqdm import tqdm

# 1) filtriraj na scFEA modulne gene (55x manjsi vhod, identicen rezultat)
mg = pd.read_csv('/content/scFEA/data/module_gene_m168.csv', index_col=0)
module_genes = set()
for col in mg.columns:
    for v in mg[col].dropna().astype(str):
        g = v.strip()
        if g and g.lower() != 'nan': module_genes.add(g)
keep_idx = [i for i, g in enumerate(gene_names) if g in module_genes]
keep_names = [gene_names[i] for i in keep_idx]
print(f'scFEA modulnih genov: {len(module_genes)} | nasih v modulih: {len(keep_idx)}')
assert len(keep_idx) > 100, 'premalo ujemanja gene-imen!'

counts_k = counts[:, keep_idx]                 # sparse, le modulni geni
N = counts_k.shape[0]
CHUNK = 5000
n_batches = (N + CHUNK - 1) // CHUNK
print(f'Celic: {N} | kosov po {CHUNK}: {n_batches}')

os.makedirs('/content/scfea_input', exist_ok=True)
os.makedirs('/content/scfea_output', exist_ok=True)
%cd /content/scFEA

flux_parts = []
t0 = time.time()
for b in tqdm(range(n_batches), desc='scFEA batchi'):
    lo, hi = b*CHUNK, min((b+1)*CHUNK, N)
    Xk = counts_k[lo:hi]
    Xk = Xk.toarray() if issparse(Xk) else np.asarray(Xk)
    Xk = np.rint(Xk).astype(np.int32)
    # CSV: geni x celice; imena celic = GLOBALNI indeks (c{lo}..c{hi-1}) -> unikatna cez kose
    df_in = pd.DataFrame(Xk.T, index=keep_names, columns=[f'c{i}' for i in range(lo, hi)])
    in_csv = '/content/scfea_input/expr.csv'
    df_in.to_csv(in_csv)
    out_csv = f'/content/scfea_output/flux_{b}.csv'
    del Xk, df_in

    # tih zagon (stdout v /dev/null; napake se vseeno pokazejo)
    rc = os.system(
        f'python src/scFEA.py --data_dir data --input_dir /content/scfea_input '
        f'--test_file expr.csv --moduleGene_file module_gene_m168.csv '
        f'--stoichiometry_matrix cmMat_c70_m168.csv '
        f'--output_flux_file {out_csv} --output_balance_file /content/scfea_output/bal_{b}.csv '
        f'--sc_imputation False > /content/scfea_output/log_{b}.txt 2>&1')
    assert rc == 0 and os.path.exists(out_csv), f'batch {b} PADEL (glej log_{b}.txt)'
    flux_parts.append(pd.read_csv(out_csv, index_col=0))

print(f'\n=== scFEA batch cas: {time.time()-t0:.0f} s za {N} celic ===')

# 2) zdruzi (stolpci=moduli so ISTI za vse kose; vrstice=celice po vrsti)
flux = pd.concat(flux_parts, axis=0)
print('Zdruzen flux:', flux.shape)

# 3) poravnaj na NAS vrstni red celic (c0..cN-1) + preveri
expected = [f'c{i}' for i in range(N)]
assert set(expected) == set(flux.index.astype(str)), 'flux celice se ne ujemajo!'
flux = flux.loc[expected]

n_nan = int(np.isnan(flux.values).sum())
print('NaN:', n_nan, '| delez nicelnih:', f'{(flux.values==0).mean():.3f}',
      '| min/max/mean:', f'{flux.values.min():.3f}/{flux.values.max():.3f}/{flux.values.mean():.3f}')
assert n_nan == 0, 'FLUX VSEBUJE NaN!'
assert flux.shape[0] == counts.shape[0], f'flux vrstic != celic!'

# 4) shrani (poravnan z data_labels po vrsticah, kot data_rna/data_tcr)
data_flux = flux.values.astype(np.float32)
with open(os.path.join(DATA, 'data_flux.pkl'), 'wb') as f:
    pickle.dump(data_flux, f)
with open(os.path.join(DATA, 'flux_module_names.pkl'), 'wb') as f:
    pickle.dump(list(flux.columns), f)
print(f'\ndata_flux.pkl shranjen: {data_flux.shape} (3. modaliteta za TRIM)')

## 5. Zakljucek

- `data_flux.pkl` (celice x ~168 modulov) = metabolni flux, poravnan z RNA/TCR.
- Naslednje: flux encoder/decoder kot 3. modaliteta v TRIM (RNA + TCR + Flux, Var 2).

Pridrzki za diplomo: scFEA fluksi RELATIVNI/model-odvisni; scFEA nevzdrzevan (patchi za pandas/torch).